In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub
import os
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

c:\Users\jb255070\Documents\NFL_python_project\nfl_ml_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# download the data from kaggle
import kagglehub

# Download latest version
dataset_path = kagglehub.dataset_download("philiphyde1/nfl-stats-1999-2022")

In [3]:
print("Files in the dataset folder:", os.listdir(dataset_path))

Files in the dataset folder: ['weekly_player_stats_defense.csv', 'weekly_player_stats_offense.csv', 'weekly_team_stats_defense.csv', 'weekly_team_stats_offense.csv', 'yearly_player_stats_defense.csv', 'yearly_player_stats_offense.csv', 'yearly_team_stats_defense.csv', 'yearly_team_stats_offense.csv']


In [21]:
# Load the CSV file into a DataFrame
# [weekly_player_stats_defense.csv'', 'weekly_player_stats_defense.csv', 'weekly_team_stats_defense.csv', 'weekly_team_stats_defense.csv', 'yearly_player_stats_defense.csv', 'yearly_player_stats_defense.csv', 'yearly_team_stats_defense.csv', 'yearly_team_stats_defense.csv']
weekly_df_defense = pd.read_csv(os.path.join(dataset_path, 'weekly_team_stats_defense.csv'))

weekly_df_offense = pd.read_csv(os.path.join(dataset_path, 'weekly_team_stats_offense.csv'))

In [22]:
weekly_df_defense.head()

,game_id,season,week,team,season_type,safety,interception,fumble,fumble_lost,fumble_forced,...,average_solo_tackle,average_assist_tackle,average_tackle_with_assist,average_sack,average_qb_hit,average_def_touchdown,average_defensive_two_point_attempt,average_defensive_two_point_conv,average_defensive_extra_point_attempt,average_defensive_extra_point_conv
0,2012_01_SEA_ARI,2012,1,ARI,REG,0,1,2,1,1,...,55.000000,6.0,4.000000,3.0,8.000000,0.000000,0.0,0.0,0,0
1,2012_02_ARI_NE,2012,2,ARI,REG,0,1,0,0,0,...,51.000000,7.0,2.000000,3.5,7.000000,0.000000,0.0,0.0,0,0
2,2012_03_PHI_ARI,2012,3,ARI,REG,0,0,2,2,2,...,49.333333,6.0,1.666667,4.0,9.333333,0.333333,0.0,0.0,0,0
3,2012_04_MIA_ARI,2012,4,ARI,REG,0,2,4,2,3,...,50.500000,6.5,2.500000,4.0,9.500000,0.250000,0.0,0.0,0,0
4,2012_05_ARI_STL,2012,5,ARI,REG,0,1,0,0,0,...,47.400000,6.2,2.000000,3.4,8.800000,0.200000,0.0,0.0,0,0


In [23]:
num_columns_df = weekly_df_defense.shape[1]
num_rows_df = weekly_df_defense.shape[0]
column_names_df = weekly_df_defense.columns.tolist()

print(f"The dataset has {num_columns_df} columns and {num_rows_df} rows")

for col in column_names_df:
    null_count = weekly_df_defense[col].isnull().sum()  # Count null values
    print(f"Column: {col}, "
          f"Type: {weekly_df_defense[col].dtype}, "
          f"Unique Values: {weekly_df_defense[col].nunique()}, "
          f"Null Values: {null_count} ({null_count/num_rows_df:.1%})")

The dataset has 65 columns and 7088 rows
Column: game_id, Type: object, Unique Values: 3544, Null Values: 0 (0.0%)
Column: season, Type: int64, Unique Values: 13, Null Values: 0 (0.0%)
Column: week, Type: int64, Unique Values: 22, Null Values: 0 (0.0%)
Column: team, Type: object, Unique Values: 32, Null Values: 0 (0.0%)
Column: season_type, Type: object, Unique Values: 2, Null Values: 0 (0.0%)
Column: safety, Type: int64, Unique Values: 3, Null Values: 0 (0.0%)
Column: interception, Type: int64, Unique Values: 7, Null Values: 0 (0.0%)
Column: fumble, Type: int64, Unique Values: 8, Null Values: 0 (0.0%)
Column: fumble_lost, Type: int64, Unique Values: 6, Null Values: 0 (0.0%)
Column: fumble_forced, Type: int64, Unique Values: 7, Null Values: 0 (0.0%)
Column: fumble_not_forced, Type: int64, Unique Values: 6, Null Values: 0 (0.0%)
Column: fumble_out_of_bounds, Type: int64, Unique Values: 5, Null Values: 0 (0.0%)
Column: solo_tackle, Type: int64, Unique Values: 56, Null Values: 0 (0.0%)
Co

In [24]:
num_columns_of = weekly_df_offense.shape[1]
num_rows_of = weekly_df_offense.shape[0]
column_names_of = weekly_df_offense.columns.tolist()

print(f"The dataset has {num_columns_of} columns and {num_rows_of} rows")

for col in column_names_of:
    null_count = weekly_df_offense[col].isnull().sum()  # Count null values
    print(f"Column: {col}, "
          f"Type: {weekly_df_offense[col].dtype}, "
          f"Unique Values: {weekly_df_offense[col].nunique()}, "
          f"Null Values: {null_count} ({null_count/num_rows_of:.1%})")

The dataset has 115 columns and 7088 rows
Column: game_id, Type: object, Unique Values: 3544, Null Values: 0 (0.0%)
Column: season, Type: int64, Unique Values: 13, Null Values: 0 (0.0%)
Column: week, Type: int64, Unique Values: 22, Null Values: 0 (0.0%)
Column: team, Type: object, Unique Values: 32, Null Values: 0 (0.0%)
Column: season_type, Type: object, Unique Values: 2, Null Values: 0 (0.0%)
Column: shotgun, Type: int64, Unique Values: 87, Null Values: 0 (0.0%)
Column: no_huddle, Type: int64, Unique Values: 70, Null Values: 0 (0.0%)
Column: qb_dropback, Type: int64, Unique Values: 63, Null Values: 0 (0.0%)
Column: qb_scramble, Type: int64, Unique Values: 12, Null Values: 0 (0.0%)
Column: total_off_yards, Type: int64, Unique Values: 461, Null Values: 0 (0.0%)
Column: pass_attempts, Type: int64, Unique Values: 62, Null Values: 0 (0.0%)
Column: complete_pass, Type: int64, Unique Values: 45, Null Values: 0 (0.0%)
Column: incomplete_pass, Type: int64, Unique Values: 32, Null Values: 0 (0

In [25]:
weekly_df_defense.head()

,game_id,season,week,team,season_type,safety,interception,fumble,fumble_lost,fumble_forced,...,average_solo_tackle,average_assist_tackle,average_tackle_with_assist,average_sack,average_qb_hit,average_def_touchdown,average_defensive_two_point_attempt,average_defensive_two_point_conv,average_defensive_extra_point_attempt,average_defensive_extra_point_conv
0,2012_01_SEA_ARI,2012,1,ARI,REG,0,1,2,1,1,...,55.000000,6.0,4.000000,3.0,8.000000,0.000000,0.0,0.0,0,0
1,2012_02_ARI_NE,2012,2,ARI,REG,0,1,0,0,0,...,51.000000,7.0,2.000000,3.5,7.000000,0.000000,0.0,0.0,0,0
2,2012_03_PHI_ARI,2012,3,ARI,REG,0,0,2,2,2,...,49.333333,6.0,1.666667,4.0,9.333333,0.333333,0.0,0.0,0,0
3,2012_04_MIA_ARI,2012,4,ARI,REG,0,2,4,2,3,...,50.500000,6.5,2.500000,4.0,9.500000,0.250000,0.0,0.0,0,0
4,2012_05_ARI_STL,2012,5,ARI,REG,0,1,0,0,0,...,47.400000,6.2,2.000000,3.4,8.800000,0.200000,0.0,0.0,0,0


In [26]:
weekly_df_offense.head()

,game_id,season,week,team,season_type,shotgun,no_huddle,qb_dropback,qb_scramble,total_off_yards,...,average_fourth_down_failed,average_rush_touchdown,average_pass_touchdown,average_safety,average_interception,average_fumble,average_fumble_lost,average_fumble_forced,average_fumble_not_forced,average_fumble_out_of_bounds
0,2012_01_SEA_ARI,2012,1,ARI,REG,31,0,38,1,258,...,0.000000,1.000000,1.000000,0.0,1.000000,2.000000,1.000000,2.000000,0.0,0.0
1,2012_02_ARI_NE,2012,2,ARI,REG,25,7,30,1,242,...,0.000000,1.000000,1.000000,0.0,0.500000,2.000000,1.500000,2.000000,0.0,0.0
2,2012_03_PHI_ARI,2012,3,ARI,REG,31,0,29,2,321,...,0.333333,0.666667,1.333333,0.0,0.333333,1.666667,1.333333,1.666667,0.0,0.0
3,2012_04_MIA_ARI,2012,4,ARI,REG,52,5,56,0,352,...,0.250000,0.500000,1.750000,0.0,0.750000,1.500000,1.000000,1.500000,0.0,0.0
4,2012_05_ARI_STL,2012,5,ARI,REG,56,7,59,1,334,...,0.600000,0.400000,1.400000,0.0,0.600000,1.400000,1.000000,1.400000,0.0,0.0


In [40]:
wdf_def_2024 = weekly_df_defense[
    (weekly_df_defense['season'].isin([2024])) ]  # Using .copy() to avoid SettingWithCopyWarning
print(f"Number of rows in 2024 season: {wdf_def_2024.head()}")

Number of rows in 2024 season:               game_id  season  week team season_type  safety  interception  \
6518  2024_01_ARI_BUF    2024     1  ARI         REG       0             0   
6519   2024_02_LA_ARI    2024     2  ARI         REG       0             0   
6520  2024_03_DET_ARI    2024     3  ARI         REG       0             1   
6521  2024_04_WAS_ARI    2024     4  ARI         REG       0             1   
6522   2024_05_ARI_SF    2024     5  ARI         REG       0             2   

      fumble  fumble_lost  fumble_forced  ...  average_solo_tackle  \
6518       1            1              1  ...            28.000000   
6519       1            1              1  ...            29.000000   
6520       0            0              0  ...            30.333333   
6521       0            0              0  ...            33.500000   
6522       1            1              1  ...            34.000000   

      average_assist_tackle  average_tackle_with_assist  average_sack  \
6518  

In [39]:
wdf_of_2024 = weekly_df_offense[
    (weekly_df_offense['season'].isin([2024])) ]
print(f"Number of rows in 2024 season: {wdf_of_2024.head()}")

Number of rows in 2024 season:               game_id  season  week team season_type  shotgun  no_huddle  \
6518  2024_01_ARI_BUF    2024     1  ARI         REG       51          4   
6519   2024_02_LA_ARI    2024     2  ARI         REG       37          3   
6520  2024_03_DET_ARI    2024     3  ARI         REG       47          8   
6521  2024_04_WAS_ARI    2024     4  ARI         REG       43         13   
6522   2024_05_ARI_SF    2024     5  ARI         REG       42          0   

      qb_dropback  qb_scramble  total_off_yards  ...  \
6518           38            3              286  ...   
6519           26            4              497  ...   
6520           37            2              284  ...   
6521           27            1              323  ...   
6522           33            2              364  ...   

      average_fourth_down_failed  average_rush_touchdown  \
6518                         1.0                2.000000   
6519                         0.5                2.00000

In [34]:
wdf_def_2024.describe()

,season,week,safety,interception,fumble,fumble_lost,fumble_forced,fumble_not_forced,fumble_out_of_bounds,solo_tackle,...,average_solo_tackle,average_assist_tackle,average_tackle_with_assist,average_sack,average_qb_hit,average_def_touchdown,average_defensive_two_point_attempt,average_defensive_two_point_conv,average_defensive_extra_point_attempt,average_defensive_extra_point_conv
count,7088.000000,7088.000000,7088.000000,7088.000000,7088.000000,7088.000000,7088.000000,7088.000000,7088.000000,7088.000000,...,7088.000000,7088.000000,7088.000000,7088.000000,7088.000000,7088.000000,7088.000000,7088.000000,7088.0,7088.0
mean,2018.091422,9.626693,0.030897,0.823505,1.228132,0.555164,0.827596,0.406603,0.110045,39.291337,...,39.454111,11.964636,4.313869,2.391020,5.348949,0.131890,0.007074,0.001221,0.0,0.0
std,3.760292,5.402212,0.173865,0.946592,1.125917,0.743955,0.920988,0.657330,0.338942,7.580297,...,4.237502,3.417597,2.841348,0.792285,1.347254,0.169108,0.031945,0.012628,0.0,0.0
min,2012.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,13.000000,...,22.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
25%,2015.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,34.000000,...,36.682292,9.642857,2.125000,1.937500,4.500000,0.000000,0.000000,0.000000,0.0,0.0
50%,2018.000000,10.000000,0.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,39.000000,...,39.250000,12.000000,4.000000,2.384615,5.285714,0.090909,0.000000,0.000000,0.0,0.0
75%,2021.000000,14.000000,0.000000,1.000000,2.000000,1.000000,1.000000,1.000000,0.000000,44.000000,...,42.000000,14.444444,6.142857,2.857143,6.105263,0.200000,0.000000,0.000000,0.0,0.0
max,2024.000000,22.000000,2.000000,6.000000,7.000000,5.000000,7.000000,5.000000,4.000000,73.000000,...,61.000000,28.000000,23.000000,10.000000,17.000000,2.000000,1.000000,0.500000,0.0,0.0


In [35]:
merged_df = pd.merge(wdf_of_2024, wdf_def_2024, on='game_id', how='outer')

In [42]:
merged_df.describe()

,season_x,week_x,shotgun,no_huddle,qb_dropback,qb_scramble,total_off_yards,pass_attempts,complete_pass,incomplete_pass,...,average_solo_tackle,average_assist_tackle,average_tackle_with_assist,average_sack,average_qb_hit,average_def_touchdown,average_defensive_two_point_attempt,average_defensive_two_point_conv,average_defensive_extra_point_attempt,average_defensive_extra_point_conv
count,1140.0,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,...,7658.000000,7658.000000,7658.000000,7658.000000,7658.000000,7658.000000,7658.000000,7658.000000,7658.0,7658.0
mean,2024.0,9.950877,46.777193,8.457895,37.085965,2.126316,354.014035,31.885965,21.291228,10.594737,...,39.130104,12.212935,4.424489,2.399433,5.350952,0.128021,0.007047,0.001381,0.0,0.0
std,0.0,5.603621,12.044890,9.387333,8.439922,1.802790,80.697855,7.710931,5.563845,4.199039,...,4.292874,3.451305,2.803911,0.788341,1.356849,0.166167,0.031526,0.013372,0.0,0.0
min,2024.0,1.000000,12.000000,0.000000,14.000000,0.000000,125.000000,10.000000,7.000000,0.000000,...,22.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
25%,2024.0,5.000000,39.000000,2.000000,31.000000,1.000000,299.000000,27.000000,17.000000,8.000000,...,36.214286,9.833333,2.268182,2.000000,4.500000,0.000000,0.000000,0.000000,0.0,0.0
50%,2024.0,10.000000,47.000000,6.000000,37.000000,2.000000,349.000000,31.000000,21.000000,10.000000,...,39.000000,12.250000,4.142857,2.400000,5.285714,0.083333,0.000000,0.000000,0.0,0.0
75%,2024.0,15.000000,54.000000,11.000000,42.000000,3.000000,411.000000,37.000000,25.000000,13.000000,...,41.833333,14.700000,6.200000,2.857143,6.111111,0.200000,0.000000,0.000000,0.0,0.0
max,2024.0,22.000000,81.000000,64.000000,63.000000,11.000000,645.000000,59.000000,42.000000,27.000000,...,61.000000,28.000000,23.000000,10.000000,17.000000,2.000000,1.000000,0.500000,0.0,0.0
